## Dict entry parser

In [1]:
from pathlib import Path
from patterns.macro_structure_patterns import VLASNI_IMENA_DICT_MACRO_PATTERN
from parsers.macro_parser import MacroParser
from utils.file_manager import FileManager
from utils import custom_logger
import re
from patterns.micro_structure_patterns import ENTRY_PATTERNS
from collections import defaultdict

In [2]:
def preprocess_entry(path: Path):
    """
    Preprocess the  file.
    """
    text = FileManager().read_md(path)
    text = re.sub(r'(\w)-\n\n(?=[А-ЯІЇЄҐЁA-Z])', r'\1', text)
    
    text = re.sub(
        r'([А-ЯІЇЄҐЁA-Z][^\n]+,)\n\n(?=[А-ЯІЇЄҐЁ][А-ЯІЇЄҐЁA-Z\`\'\w\u0301,;\s]+\.)',
        r'\1\n',
        text
    )
    FileManager().write_md(text, path.parent, name=path.name)

In [3]:
fem_file = Path("/workspaces/Ukrainian-Dictionary-Parser/data/dictionary/dictionary_f.md")
male_file = Path("/workspaces/Ukrainian-Dictionary-Parser/data/dictionary/dictionary_m.md")

preprocess_entry(fem_file)
preprocess_entry(male_file)

In [4]:
parser = MacroParser()

In [5]:
parser.parse_dict(file_path=fem_file, output_path="/workspaces/Ukrainian-Dictionary-Parser/data", pattern=VLASNI_IMENA_DICT_MACRO_PATTERN, name="fem_parsed.jsonl", replace_md=True)
parser.parse_dict(file_path=male_file, output_path="/workspaces/Ukrainian-Dictionary-Parser/data", pattern=VLASNI_IMENA_DICT_MACRO_PATTERN, name="male_parsed.jsonl",  replace_md=True)

[2026-05-14 11:37:06][File Manager][file_manager.py:38][INFO] Successfully saved file fem_parsed.jsonl in /workspaces/Ukrainian-Dictionary-Parser/data
[2026-05-14 11:37:06][File Manager][file_manager.py:38][INFO] Successfully saved file male_parsed.jsonl in /workspaces/Ukrainian-Dictionary-Parser/data


## Dict entry structure parser

In [1]:
from patterns.micro_structure_patterns import ENTRY_PATTERNS
from parsers.micro_parser import MicroParser

In [6]:
fm = FileManager()
micro_parser = MicroParser()
male_jsonl = Path("/workspaces/Ukrainian-Dictionary-Parser/data/male_parsed.jsonl")
female_jsonl = Path("/workspaces/Ukrainian-Dictionary-Parser/data/fem_parsed.jsonl")

In [7]:
micro_parser.parse_dict_entries_multi(male_jsonl, Path("data/test_patterns/output_parsed_entries"), ENTRY_PATTERNS, "male_entries_parsed_V2.jsonl")

[2026-05-14 15:25:27][Micro Parser][micro_parser.py:127][INFO] Total entries to parse: 458
[2026-05-14 15:25:27][File Manager][file_manager.py:38][INFO] Successfully saved file male_entries_parsed_V2.jsonl in data/test_patterns/output_parsed_entries
[2026-05-14 15:25:27][Micro Parser][micro_parser.py:146][INFO] Parsed 441/458 entries → data/test_patterns/output_parsed_entries/male_entries_parsed_V2.jsonl
[2026-05-14 15:25:27][Micro Parser][micro_parser.py:149][INFO] Pattern breakdown: {'off_etym': 23, 'unofficial_vars_only': 3, 'ref_eq': 56, 'semi_off_etym_unoffic': 11, 'etym_unoffic': 148, 'off_etym_unoffic': 48, 'etym': 92, 'off_half_etym': 3, 'off_half_etym_unoffic': 19, 'half_etym': 9, 'fem_free_text_unoffic': 2, 'ref_etym_same': 7, 'half_etym_unoffic': 15, 'semi_off_etym': 4, 'ref_numbered': 1}
[2026-05-14 15:25:27][File Manager][file_manager.py:38][INFO] Successfully saved file unmatched_male_entries_parsed_V2.jsonl in data/test_patterns/output_parsed_entries
[2026-05-14 15:25:27

## Preprocessing entries

In [2]:
from copy import deepcopy
from abbreviations import abbreviations
import re

In [104]:
parsed_entries_male = fm.jsonl2dict(Path("data/parsed_entries/male_entries_parsed_V2.jsonl"))
parsed_entries_female = fm.jsonl2dict(Path("data/parsed_entries/female_entries_parsed_V2.jsonl"))

In [92]:
def preprocess(path: Path, output_path:Path, name:str=None) -> dict:
    output_name = name if name else path.name
    data = FileManager().jsonl2dict(path)
    preprocessed_data = list()
    for entry in data:
        preprocessed_dict = deepcopy(entry)
        text_data = {key: re.sub(r"\.;?$", "", value) for key, value in preprocessed_dict.items() if isinstance(value, str)}
        name_text_data = {key: re.sub(r"\s?-", "", value) for key, value in preprocessed_dict.items() if key not in ["entry_id", "etymology", "etymology_comment"] and isinstance(value, str)}
        preprocessed_dict.update(text_data)
        preprocessed_dict.update(name_text_data)
        preprocessed_data.append(preprocessed_dict)
    FileManager().dict2jsonl(preprocessed_data, output_path, name=output_name)

In [126]:
def replace_abbreviations(path:Path, output_path:Path, abbreviations: dict, ignore: list = None, name:str=None) -> dict:
    output_name = name if name else path.name
    data = FileManager().jsonl2dict(path)
    preprocessed_data = list()
    for entry in data:
        if ignore is None:
            ignore = []

        ignore_set = set(ignore)

        sorted_keys = sorted(
            (k for k in abbreviations if k not in ignore_set),
            key=len,
            reverse=True,
        )

        def expand(text: str) -> str:
            for key in sorted_keys:
                pattern = r'(?<!\w)' + re.escape(key) + r'\.?(?!\w)'
                text = re.sub(pattern, abbreviations[key], text)
            return text

        preprocessed_data.append({k: expand(v) if isinstance(v, str) else v for k, v in entry.items()})
    FileManager().dict2jsonl(preprocessed_data, output_path, name=output_name)

In [95]:
preprocess(path=Path("data/parsed_entries/male_entries_parsed_V2.jsonl"), output_path=Path("data/parsed_entries/male_entries_parsed_V2.jsonl").parent)

[2026-05-15 18:00:32][File Manager][file_manager.py:38][INFO] Successfully saved file male_entries_parsed_V2.jsonl in data/parsed_entries


In [132]:
replace_abbreviations(path=Path("data/parsed_entries/female_entries_parsed_V2.jsonl"), output_path=Path("data/parsed_entries/female_entries_parsed_V2.jsonl").parent, abbreviations=abbreviations, ignore=["entry_id", "etymology_comment"])

[2026-05-15 18:44:18][File Manager][file_manager.py:38][INFO] Successfully saved file female_entries_parsed_V2.jsonl in data/parsed_entries


## Name transformer

In [2]:
from utils import name_transformer

In [3]:
name_transformer.NameTransformer.build_combined(male_filepath="data/parsed_entries/male_entries_parsed_V2.jsonl", female_filepath="data/parsed_entries/female_entries_parsed_V2.jsonl", output_path="data_transformed.jsonl")

Saved 905 records (498 male + 407 female) -> data_transformed.jsonl


[{'entry_id': 1,
  'name': 'Абакум',
  'description': {'sex': 'male',
   'base_name': '',
   'etymology': 'давньоєврейське',
   'etymology_comment': 'Chābhaqqūq — ім’я біблійного пророка; від chābhaqqūq — обійми (Божі)'},
  'variant_info': {'official_vars': ['Авакум'],
   'unofficial_vars': ['Абакумко',
    'Абакумонько',
    'Абакумочко',
    'Авакумко',
    'Авакумонько',
    'Авакумочко',
    'Бакум']}},
 {'entry_id': 2,
  'name': 'Абрам',
  'description': {'sex': 'male',
   'base_name': '',
   'etymology': 'давньоєврейське',
   'etymology_comment': 'ім’я ’Abhrām — отець піднесений; пізніше ’Abhrāhām — батько багатьох [народів]'},
  'variant_info': {'official_vars': ['Аврам', 'Оврам'],
   'unofficial_vars': []}},
 {'entry_id': 3,
  'name': 'Авакум',
  'description': {'sex': 'male',
   'base_name': '',
   'etymology': 'давньоєврейське',
   'etymology_comment': 'Chābhaqqūq — ім’я біблійного пророка; від chābhaqqūq — обійми (Божі)'},
  'variant_info': {'official_vars': ['Абакум'], 'uno

In [38]:
transformed_fem_df[
    transformed_fem_df["etymology"].isnull()
]

,entry_id,name,official_vars,base_name,unofficial_vars,etymology,etymology_comment
189,190,ЛЕСЯ,[],NaN,"[ЛЕСЕНЬКА, ЛЕСЕЧКА, ЛЕСЮНЯ, ЛЕСЮНЕНЬКА, ЛЕСЮНЕ...",NaN,"скорочений варіант від ряду імен (Лариса, Олек..."
269,270,ОЛЕСЯ,[],NaN,[],NaN,скорочений варіант імені Олександра. Фіксуєтьс...
292,293,ПОЛІНА,[],NaN,"[ПОЛІНОНЬКА, ПОЛІНОЧКА, ПОЛІНКА, ПОЛЯ, ПОЛЕНЬК...",NaN,"скорочений варіант імені Аполлінарія, що став ..."
295,296,РАЇНА,[],NaN,"[РАЇНОНЬКА, РАЇНОЧКА, РАЯ]",NaN,"можливо, фонетичний варіант імені Раїса"
399,400,РАФАЛІНА,[],NaN,"[РАФАЛЯ, РАФЕЛЯ, ФАЛЯ, ФЕЛЯ]",NaN,"утворене від чоловічий рід, чоловіче ім'я імен..."


In [39]:
transformed_male_df[transformed_male_df["etymology"].isnull()]

,entry_id,name,official_vars,etymology,etymology_comment,unofficial_vars
102,103,ГЛІБ,[],NaN,"вважають, що це раннє запозичення із скандинав...","[ГЛІБКО, ГЛІБОНЬКО, ГЛІБОЧКО, ГЛІБЦЬО]"
471,472,ЛЕСЬ,[],NaN,"скорочений, скорочення варіант ряду імен (Олек...",[]
479,480,ОЛЕСЬ,[],NaN,"скорочений, скорочення варіант від імені Олекс...",[]


## Transcriptor

In [2]:
from utils import transliterater

In [4]:
transliterator = transliterater.Transliterator()
transliterator.transliterate_names(input_path=Path("male_data_transformed.jsonl"), output_path=Path("/workspaces/Ukrainian-Dictionary-Parser"), name="male_entries_transliterated.jsonl", target_keys=("name", "official_vars", "unofficial_vars"))

[2026-05-23 18:06:41][File Manager][file_manager.py:38][INFO] Successfully saved file male_entries_transliterated.jsonl in /workspaces/Ukrainian-Dictionary-Parser


In [5]:
import json
import sys
 
 
def iter_records(path):
    """Читає JSONL-файл порядково, пропускаючи порожні рядки."""
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise SystemExit(f"Помилка JSON у {path}, рядок {line_no}: {e}")
 
 
def merge(first_path, second_path, output_path):
    next_id = 1
    written = 0
 
    with open(output_path, "w", encoding="utf-8") as out:
        for src in (first_path, second_path):
            for rec in iter_records(src):
                rec["id"] = next_id          # змінюємо лише id
                next_id += 1
                # ensure_ascii=False — зберігаємо кирилицю та діакритику як є
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                written += 1
 
    print(f"Готово: {written} записів записано у {output_path} (id 1..{written}).")
 

In [6]:
merge("male_entries_transliterated.jsonl", "female_entries_transliterated.jsonl", "data_transliterated.jsonl")

Готово: 905 записів записано у data_transliterated.jsonl (id 1..905).
